# Pydantic

Pydantic validates data at runtime using Python type annotations. It is especially useful when data comes from outside your program, such as JSON, environment variables, configuration files, or API requests. A Pydantic model checks the input, converts compatible values when allowed, and gives your code a typed Python object.

This notebook uses **Pydantic v2** APIs. 

## What Pydantic helps with

- Validate input types and custom business rules.
- Convert compatible input values in its default lax mode, or reject conversions in strict mode.
- Represent structured data with models, nested models, unions, and standard Python types.
- Serialize validated data to dictionaries and JSON.
- Report validation failures as structured `ValidationError` details.

Pydantic is a validation library, not a database, web framework, or replacement for static type checkers. Frameworks such as FastAPI use it to validate request and response data.

In [21]:
# Install Pydantic into the environment used by this notebook, if it is not installed.
%pip install "pydantic>=2,<3"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Core Concepts
Pydantic has some core concepts that make it easy to work with data models. Some of them are:

- Models: classes inheriting BaseModel that declare fields with type annotations.
- Validation: when you create a model instance, Pydantic validates and coerces input into the declared types.
- Parsing and Serialization: Models can parse from JSON and dump back to native Python objects or JSON.
- Validators: field-level (@field_validator) and model-level (@model_validator) hooks for custom logic.
- Errors: validation failures raise ValidationError with structured details you can return to clients.

In [22]:
# A model declares the shape and types of the data it accepts.
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: int
    email: str  # str checks the type; it does not verify email-address syntax.

user = UserProfile(name="Rohit", age=33, email="rohit@example.com")
print(user)
print(user.age)  # Access model fields like ordinary object attributes.

user.age = 32  # This valid assignment works, but assignment is not revalidated by default.
print(user.age)

name='Rohit' age=33 email='rohit@example.com'
33
32


In [23]:
# Required fields and defaults
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: int = 33
    email: str
    is_active: bool = True

user = UserProfile(name="Rohit", email="rohit@example.com")
print(user)

name='Rohit' age=33 email='rohit@example.com' is_active=True


## advanced validation techniques

### Field Validators (@field_validator)
Field validators let you define custom validation rules for individual fields. They are useful when you need to enforce constraints beyond type checking.

In [24]:
from pydantic import BaseModel, field_validator

class UserProfile(BaseModel):
    name: str
    age: int
    email: str

    @field_validator("age")
    @classmethod
    def check_age(cls, value: int) -> int:
        # Raise ValueError when the value violates the model's business rule.
        if value < 18:
            raise ValueError("Age must be at least 18")
        return value

user = UserProfile(name="Amar", age=18, email="amar@example.com")
print(user)

name='Amar' age=18 email='amar@example.com'


### Validators for Entire Model (@model_validator)
Model validators let you define validation rules that involve multiple fields or the model as a whole.

In [41]:
from pydantic import BaseModel, model_validator

class User(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def passwords_match(self):
        # An after-model validator receives the fully validated model instance.
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

user = User(password="example-password", confirm_password="example-password")
print(user)

password='example-password' confirm_password='example-password'


###  Nested Models and Complex Data Structures
Nested models allow Pydantic to handle hierarchical or complex data structures cleanly.

In [42]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str

class User(BaseModel):
    name: str
    age: int
    address : Address

add = Address(street="high", city="mumbai")
usr = User(name="Rohit", age=32, address=add)
usr

User(name='Rohit', age=32, address=Address(street='high', city='mumbai'))

## Data Parsing and Serialization
Pydantic makes it easy to parse data from different formats and serialize it back to JSON or other formats.

### 1. Parsing Data from JSON and Other Formats
Pydantic can parse JSON or other serialized data formats into Python objects automatically, validating types in the process.

In [43]:
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: int
    email: str

data = '{"name": "Noah Kim", "age": 28, "email": "noah.kim@example.kr"}'
user = UserProfile.model_validate_json(data)
print(user)

name='Noah Kim' age=28 email='noah.kim@example.kr'


### 2. Serializing Models to JSON
You can convert a Pydantic model back to JSON for storage, APIs or communication with other systems.

In [44]:
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: int
    email: str

user = UserProfile(name="Luca Rossi", age=29, email="luca.rossi@example.it")
json_data = user.model_dump_json()
print(json_data)

{"name":"Luca Rossi","age":29,"email":"luca.rossi@example.it"}


### 3. Optional and nullable fields

A field typed as `int | None` (or `Optional[int]`) may contain an integer or `None`. In Pydantic v2, that annotation alone does **not** make the field omittable: without a default, the field is still required. Use `= None` when the field may be left out. A field can also be required but nullable by writing `field: int | None` without a default.

In [50]:
from typing import Optional
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: Optional[int] = None
    email: str
    add: str = 'None'
user = UserProfile(name="Mia Johansson", email="mia.johansson@example.se", add="Pune")
print(user)

name='Mia Johansson' age=None email='mia.johansson@example.se' add='Pune'


## Field constraints and model configuration

`Field()` adds constraints and metadata to a field. `ConfigDict` configures behavior for the whole model. In this example, extra input keys are rejected, strings are stripped, and assignments are validated. By default, Pydantic ignores extra input keys and does not revalidate later assignments.

In [52]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class InventoryItem(BaseModel):
    model_config = ConfigDict(
        extra="forbid",  # Reject keys that are not declared as fields.
        validate_assignment=True,  # Validate changes made after construction.
        str_strip_whitespace=True,
    )

    name: str = Field(min_length=1, description="Display name of the item")
    quantity: int = Field(ge=0, description="Units currently in stock")
    unit_price: float = Field(gt=0)

item = InventoryItem(name="  Notebook  ", quantity="4", unit_price=2.5)
print(item)  # Whitespace is stripped and the compatible string is converted to int.

try:
    item.quantity = -1
except ValidationError as error:
    print("Assignment rejected:", error.errors()[0]["msg"])

try:
    InventoryItem(name="Pen", quantity=3, unit_price=1.0, color="blue")
except ValidationError as error:
    print("Extra field rejected:", error.errors()[0]["type"])

name='Notebook' quantity=4 unit_price=2.5
Assignment rejected: Input should be greater than or equal to 0
Extra field rejected: extra_forbidden


## Lax and strict validation

Pydantic uses lax validation by default: it converts some compatible input values, such as the string `"42"` to the integer `42`. Strict mode disables many such conversions. Strictness can be set for an entire model or selected fields; the example below compares model-level behavior.

In [53]:
from pydantic import BaseModel, ConfigDict, ValidationError

class LaxRecord(BaseModel):
    count: int

class StrictRecord(BaseModel):
    model_config = ConfigDict(strict=True)
    count: int

print(LaxRecord(count="42").count)  # Converted to the integer 42.

try:
    StrictRecord(count="42")
except ValidationError as error:
    print("Strict mode rejected the string:", error.errors()[0]["type"])

print(StrictRecord(count=42))

42
Strict mode rejected the string: int_type
count=42


## Understanding validation errors

Invalid input raises `pydantic.ValidationError`. In applications, you can catch it at the input boundary and inspect `errors()` for structured details such as the failing field (`loc`), error category (`type`), and readable explanation (`msg`). Avoid exposing sensitive input values in public error responses.

In [56]:
from pydantic import BaseModel, Field, ValidationError

class Booking(BaseModel):
    guest_name: str = Field(min_length=1)
    nights: int = Field(ge=1, le=30)

try:
    Booking(guest_name="", nights=0)
except ValidationError as error:
    for issue in error.errors():
        # loc identifies the field; msg explains the violated rule.
        print(f"Field {issue['loc']}: {issue['msg']} (type={issue['type']})")

Field ('guest_name',): String should have at least 1 character (type=string_too_short)
Field ('nights',): Input should be greater than or equal to 1 (type=greater_than_equal)


## Aliases and serialization options

Aliases map an external field name (for example, JSON `userId`) to a Python-friendly attribute (`user_id`). `model_dump()` returns Python data; `model_dump_json()` returns a JSON string. Options such as `by_alias`, `exclude_none`, and `exclude` let you shape output for a specific boundary.

In [33]:
from typing import Optional
from pydantic import BaseModel, ConfigDict, Field

class ApiUser(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    user_id: int = Field(alias="userId")
    display_name: str
    nickname: Optional[str] = None
    access_token: str

# Accept either the API alias or Python field name because populate_by_name=True.
api_user = ApiUser(userId=7, display_name="Mina", access_token="demo-token")
print(api_user.user_id)
print(api_user.model_dump(by_alias=True, exclude={"access_token"}, exclude_none=True))
print(api_user.model_dump_json(by_alias=True, exclude={"access_token"}, exclude_none=True))

7
{'userId': 7, 'display_name': 'Mina'}
{"userId":7,"display_name":"Mina"}


## Validating data without a `BaseModel`

`TypeAdapter` validates and serializes a type directly. It is useful when the shape is a list, mapping, union, or another type that does not need its own model class.

In [34]:
from typing import List
from pydantic import BaseModel, TypeAdapter

class Tag(BaseModel):
    name: str

# Reuse one adapter to validate a list of tag dictionaries.
tags_adapter = TypeAdapter(List[Tag])
tags = tags_adapter.validate_python([
    {"name": "python"},
    {"name": "validation"},
])
print(tags)
print(tags_adapter.dump_json(tags).decode("utf-8"))

[Tag(name='python'), Tag(name='validation')]
[{"name":"python"},{"name":"validation"}]


## Computed fields

Use `@computed_field` when a value should be derived from validated model fields and included in serialization. A computed field is output data; it is not an additional input requirement.

In [35]:
from pydantic import BaseModel, computed_field

class CartLine(BaseModel):
    item_name: str
    unit_price: float
    quantity: int

    @computed_field
    @property
    def total(self) -> float:
        return self.unit_price * self.quantity

line = CartLine(item_name="Notebook", unit_price=2.5, quantity=4)
print(line.total)
print(line.model_dump())  # Includes the computed total.

10.0
{'item_name': 'Notebook', 'unit_price': 2.5, 'quantity': 4, 'total': 10.0}


## Before and after field validators

An `after` field validator receives a value after Pydantic has parsed it to the annotated type; this is the default mode. A `before` validator receives raw input first, which is useful for normalizing common external formats. Keep validators focused and return the value that should continue through validation.

In [36]:
from typing import Any
from pydantic import BaseModel, field_validator

class ProductCode(BaseModel):
    code: str

    @field_validator("code", mode="before")
    @classmethod
    def normalize_code(cls, value: Any) -> Any:
        # Normalize raw string input before Pydantic validates it as str.
        if isinstance(value, str):
            return value.strip().upper()
        return value

product_code = ProductCode(code="  ab-123  ")
print(product_code.code)

AB-123


## Validating function arguments

`@validate_call` applies Pydantic validation to a function's arguments. It is handy at a boundary where a helper may be called with untrusted or loosely typed values; ordinary Python type annotations alone do not validate calls.

In [37]:
from typing_extensions import Annotated
from pydantic import Field, ValidationError, validate_call

@validate_call
def repeat_message(message: str, times: Annotated[int, Field(gt=0)]) -> str:
    return " ".join([message] * times)

print(repeat_message("hello", "3"))  # Lax mode converts the compatible string to int.

try:
    repeat_message("hello", 0)
except ValidationError as error:
    print("Invalid call:", error.errors()[0]["msg"])

hello hello hello
Invalid call: Input should be greater than 0


## Practical checklist

- Use `BaseModel` for structured data and nested models for nested objects.
- Use `Field()` for simple constraints; use validators for rules that need custom logic.
- Remember that lax mode may coerce input, `Optional[T]` does not imply a default, and assignment is not revalidated unless configured.
- Catch `ValidationError` at input boundaries and inspect its structured errors.
- Use `model_dump()` for Python data and `model_dump_json()` for JSON text; explicitly exclude secrets.
- Use `TypeAdapter` for validating a type that does not need a model class.
- Pydantic validation does not replace authorization checks, database constraints, or domain-level decisions.

Some features are separate optional packages: `EmailStr` needs `email-validator`, while settings management is provided by `pydantic-settings`.